# Supply Network Design — DataCo Global Supply Chain

## Problem Description

DataCo Global is a multi-market retailer that ships products from regional **production/distribution markets** ("factories"), through **regional distribution centers** ("depots"), out to its **end-customer countries**. This notebook builds a Supply Network Design optimization model — in the style of the classic Williams "Supply Network Design" case — but with every input (capacities, demand, and shipping costs) *derived directly from the `DataCoSupplyChainDataset.csv`* transactional dataset (180,519 order lines) instead of textbook numbers.

We restrict the network to the two largest markets and the six regional depots they feed, which in turn each primarily serve one large destination country — this mirrors the structure of the original 2-factory / 6-depot / 6-customer problem almost exactly, which makes it a natural fit for the same optimization formulation:

* **Factories** = `Market` : **Europe**, **LATAM** (the two largest markets by shipped volume)
* **Depots** = `Order Region` : **Western Europe, Northern Europe, Southern Europe** (fed by Europe) and **Central America, South America, Caribbean** (fed by LATAM)
* **Customers** = the largest `Order Country` within each of those six regions: **Francia, Reino Unido, Italia, México, Brasil, República Dominicana**

Because the real dataset has no "factory capacity" or "facility cost" fields, we *derive* sensible proxies from the data (documented at each step) rather than inventing numbers arbitrarily:

| Model quantity | Derived from the dataset |
| --- | --- |
| Factory supply capacity | Total historical `Order Item Quantity` shipped by the `Market`, +5% growth buffer |
| Depot throughput capacity | Total historical `Order Item Quantity` that passed through the `Order Region` |
| Customer demand | Total historical `Order Item Quantity` ordered by the `Order Country` |
| Shipping cost (factory→depot, depot→customer) | Mean per-unit cost, computed as `(Order Item Total − Order Profit Per Order) / Order Item Quantity`, averaged over all matching order lines |
| Depot opening (fixed) cost | 10% of the region's total historical `Order Item Discount` spend, used as a proxy for the fixed cost of running that regional distribution center |
| Direct factory→customer shipping | Not present in the data (DataCo always routes through a region), so we model it as an *express/bypass* option priced at a 30% premium over the equivalent two-leg (factory→depot→customer) cost — this gives the optimizer a real alternative to routing through a depot, exactly as the original problem allowed some customers to be served directly from a factory |

We keep the same three business questions as the original case, translated to this network:

1. **Which four of the six regional depots should stay open?** (DataCo's ops team wants to consolidate from 6 to at most 4 regional distribution centers.)
2. **Should the Western Europe depot's capacity be expanded** (by 20%, for a fixed cost)?
3. **Which depots (or direct routes) should be used to satisfy each country's demand?**

---
## Model Formulation

### Sets and Indices

$f \in \text{Factories} = \{\text{Europe}, \text{LATAM}\}$

$d \in \text{Depots} = \{\text{Western Europe}, \text{Northern Europe}, \text{Southern Europe}, \text{Central America}, \text{South America}, \text{Caribbean}\}$

$c \in \text{Customers} = \{\text{Francia}, \text{Reino Unido}, \text{Italia}, \text{México}, \text{Brasil}, \text{República Dominicana}\}$

$\text{Cities} = \text{Factories} \cup \text{Depots} \cup \text{Customers}$

### Parameters

$\text{cost}_{s,t} \in \mathbb{R}^+$ : per-unit cost of shipping from source $s$ to destination $t$ (data-derived, see table above)

$\text{supply}_f \in \mathbb{R}^+$ : maximum output of factory $f$

$\text{through}_d \in \mathbb{R}^+$ : maximum throughput of depot $d$

$\text{demand}_c \in \mathbb{R}^+$ : demand of customer $c$

$\text{opencost}_d \in \mathbb{R}^+$ : fixed cost of operating depot $d$

### Decision Variables

$\text{flow}_{s,t} \geq 0$ : quantity shipped from source $s$ to destination $t$

$\text{open}_d \in \{0,1\}$ : 1 if depot $d$ stays open

$\text{expand} \in \{0,1\}$ : 1 if Western Europe's capacity is expanded by 20%

### Objective

$$\text{Minimize} \quad Z = \sum_{(s,t)} \text{cost}_{s,t}\cdot\text{flow}_{s,t} \;+\; \sum_{d} \text{opencost}_d \cdot \text{open}_d \;+\; \text{expand\_cost}\cdot\text{expand}$$

### Constraints

**Factory output** — flow leaving a factory cannot exceed its supply:
$$\sum_{t}\text{flow}_{f,t} \leq \text{supply}_f \quad \forall f$$

**Customer demand** — flow arriving at a customer must equal its demand:
$$\sum_{s}\text{flow}_{s,c} = \text{demand}_c \quad \forall c$$

**Depot flow conservation** — flow in equals flow out for every depot:
$$\sum_{s}\text{flow}_{s,d} = \sum_{t}\text{flow}_{d,t} \quad \forall d$$

**Depot capacity** — throughput is capped, and is zero if the depot is closed (except Western Europe, which can also be expanded):
$$\sum_{s}\text{flow}_{s,d} \leq \text{through}_d \cdot \text{open}_d \quad \forall d \neq \text{Western Europe}$$
$$\sum_{s}\text{flow}_{s,\text{Western Europe}} \leq \text{through}_{\text{Western Europe}} \cdot (1 + 0.2\cdot\text{expand})$$

**Depot count** — at most 4 of the 6 depots may remain open, and the two largest (Western Europe, Central America) are always kept open:
$$\sum_{d}\text{open}_d \leq 4, \qquad \text{open}_{\text{Western Europe}} = \text{open}_{\text{Central America}} = 1$$

---
## Python Implementation

We use **PuLP** (open-source MILP solver, CBC backend) so the notebook runs anywhere without a commercial license.

In [ ]:
%pip install pulp

In [2]:
import pandas as pd
import pulp

# tested with Python 3.11 & PuLP 2.x (CBC solver)

## Step 1 — Load and Inspect the Data

In [3]:
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')
print(df.shape)
df[['Market','Order Region','Order Country','Order Item Quantity',
    'Order Item Total','Order Profit Per Order','Order Item Discount']].head()

(180519, 53)


,Market,Order Region,Order Country,Order Item Quantity,Order Item Total,Order Profit Per Order,Order Item Discount
0,Pacific Asia,Southeast Asia,Indonesia,1,314.640015,91.250000,13.110000
1,Pacific Asia,South Asia,India,1,311.359985,-249.089996,16.389999
2,Pacific Asia,South Asia,India,1,309.720001,-247.779999,18.030001
3,Pacific Asia,Oceania,Australia,1,304.809998,22.860001,22.940001
4,Pacific Asia,Oceania,Australia,1,298.250000,134.210007,29.500000


## Step 2 — Define the Network Structure

We select the two largest markets, the six depot regions they feed, and each region's largest destination country as the modeled customer.

In [4]:
markets = ['Europe', 'LATAM']

regions = ['Western Europe', 'Northern Europe', 'Southern Europe',
           'Central America', 'South America', 'Caribbean']

# each depot's largest destination country becomes the modeled customer
region_to_customer = {
    'Western Europe':   'Francia',
    'Northern Europe':  'Reino Unido',
    'Southern Europe':  'Italia',
    'Central America':  'México',
    'South America':    'Brasil',
    'Caribbean':        'República Dominicana',
}
customers = list(region_to_customer.values())

# which market feeds which depot
region_to_market = {
    'Western Europe': 'Europe', 'Northern Europe': 'Europe', 'Southern Europe': 'Europe',
    'Central America': 'LATAM', 'South America': 'LATAM', 'Caribbean': 'LATAM',
}

forced_open = ['Western Europe', 'Central America']   # always-open depots (largest two)
optional_depots = [r for r in regions if r not in forced_open]
expand_depot = 'Western Europe'
expand_cost = 15000        # assumption: analogous to the $3,000 Birmingham-expansion cost in the classic case
expand_pct = 0.20          # 20% capacity increase
direct_premium = 1.30      # assumption: 30% cost premium for bypassing the regional depot

print('Depots:', regions)
print('Customers:', region_to_customer)

Depots: ['Western Europe', 'Northern Europe', 'Southern Europe', 'Central America', 'South America', 'Caribbean']
Customers: {'Western Europe': 'Francia', 'Northern Europe': 'Reino Unido', 'Southern Europe': 'Italia', 'Central America': 'México', 'South America': 'Brasil', 'Caribbean': 'República Dominicana'}


## Step 3 — Derive Supply, Throughput, Demand, and Fixed Costs from the Data

In [5]:
# per-unit cost proxy: (revenue - profit) / quantity = cost incurred per unit shipped
df['unit_cost'] = (df['Order Item Total'] - df['Order Profit Per Order']) / df['Order Item Quantity'].replace(0, 1)

# Factory supply capacity: total historical volume shipped by the market, +5% growth buffer
supply = (df[df['Market'].isin(markets)]
          .groupby('Market')['Order Item Quantity'].sum() * 1.05).round(-2).to_dict()

# Depot throughput capacity: total historical volume through the region
through = (df[df['Order Region'].isin(regions)]
           .groupby('Order Region')['Order Item Quantity'].sum()).round(-2).to_dict()

# Customer demand: total historical volume ordered by the country
demand = (df[df['Order Country'].isin(customers)]
          .groupby('Order Country')['Order Item Quantity'].sum()).round(-2).to_dict()

# Depot fixed (opening) cost: 10% of the region's historical discount spend, as a facility-cost proxy
opencost = (df[df['Order Region'].isin(regions)]
            .groupby('Order Region')['Order Item Discount'].sum() / 10).round(-2).to_dict()

print('Factory supply  :', supply)
print('Depot throughput:', through)
print('Customer demand :', demand)
print('Depot opencost  :', opencost)

Factory supply  : {'Europe': 110500.0, 'LATAM': 118600.0}
Depot throughput: {'Caribbean': 18200, 'Central America': 62100, 'Northern Europe': 20400, 'South America': 32600, 'Southern Europe': 19500, 'Western Europe': 56500}
Customer demand : {'Brasil': 17400, 'Francia': 27700, 'Italia': 10200, 'México': 28800, 'Reino Unido': 15200, 'República Dominicana': 8100}
Depot opencost  : {'Caribbean': 16900.0, 'Central America': 57200.0, 'Northern Europe': 21600.0, 'South America': 30100.0, 'Southern Europe': 21000.0, 'Western Europe': 59800.0}


In [6]:
# Shipping cost: factory -> depot (mean per-unit cost over matching order lines)
fd_cost = (df[(df['Market'].isin(markets)) & (df['Order Region'].isin(regions))]
           .groupby(['Market', 'Order Region'])['unit_cost'].mean()).round(2).to_dict()

# Shipping cost: depot -> customer
dc_cost = (df[(df['Order Region'].isin(regions)) & (df['Order Country'].isin(customers))]
           .groupby(['Order Region', 'Order Country'])['unit_cost'].mean()).round(2).to_dict()

# Direct factory -> customer "express" cost: 30% premium over the equivalent two-leg route
direct_cost = {}
for region, cust in region_to_customer.items():
    mkt = region_to_market[region]
    two_leg = fd_cost[(mkt, region)] + dc_cost[(region, cust)]
    direct_cost[(mkt, cust)] = round(two_leg * direct_premium, 2)

print('Factory -> Depot cost   :', fd_cost)
print('Depot -> Customer cost  :', dc_cost)
print('Direct Factory->Customer:', direct_cost)

Factory -> Depot cost   : {('Europe', 'Northern Europe'): 124.75, ('Europe', 'Southern Europe'): 123.31, ('Europe', 'Western Europe'): 123.63, ('LATAM', 'Caribbean'): 105.75, ('LATAM', 'Central America'): 105.65, ('LATAM', 'South America'): 104.54}
Depot -> Customer cost  : {('Caribbean', 'República Dominicana'): 103.92, ('Central America', 'México'): 105.99, ('Northern Europe', 'Reino Unido'): 124.56, ('South America', 'Brasil'): 105.48, ('Southern Europe', 'Italia'): 123.54, ('Western Europe', 'Francia'): 122.8}
Direct Factory->Customer: {('Europe', 'Francia'): 320.36, ('Europe', 'Reino Unido'): 324.1, ('Europe', 'Italia'): 320.91, ('LATAM', 'México'): 275.13, ('LATAM', 'Brasil'): 273.03, ('LATAM', 'República Dominicana'): 272.57}


### Input Data Summary

In [7]:
summary = pd.DataFrame({
    'Depot': regions,
    'Fed by market': [region_to_market[r] for r in regions],
    'Serves customer': [region_to_customer[r] for r in regions],
    'Throughput cap.': [through[r] for r in regions],
    'Open cost ($)': [opencost[r] for r in regions],
    'Forced open?': [r in forced_open for r in regions],
})
summary

,Depot,Fed by market,Serves customer,Throughput cap.,Open cost ($),Forced open?
0,Western Europe,Europe,Francia,56500,59800.0,True
1,Northern Europe,Europe,Reino Unido,20400,21600.0,False
2,Southern Europe,Europe,Italia,19500,21000.0,False
3,Central America,LATAM,México,62100,57200.0,True
4,South America,LATAM,Brasil,32600,30100.0,False
5,Caribbean,LATAM,República Dominicana,18200,16900.0,False


## Step 4 — Model Deployment

We create the PuLP model, decision variables, objective, and constraints, following the formulation above.

In [8]:
model = pulp.LpProblem('SupplyNetworkDesign_DataCo', pulp.LpMinimize)

arcs_fd = list(fd_cost.keys())        # factory -> depot arcs
arcs_dc = list(dc_cost.keys())        # depot -> customer arcs
arcs_direct = list(direct_cost.keys())# factory -> customer (direct/express) arcs

flow_fd = pulp.LpVariable.dicts('flow_fd', arcs_fd, lowBound=0)
flow_dc = pulp.LpVariable.dicts('flow_dc', arcs_dc, lowBound=0)
flow_direct = pulp.LpVariable.dicts('flow_direct', arcs_direct, lowBound=0)

open_d = pulp.LpVariable.dicts('open', regions, cat='Binary')
expand = pulp.LpVariable('expand', cat='Binary')

# Western Europe and Central America are always open
for r in forced_open:
    open_d[r].lowBound = 1
    open_d[r].upBound = 1

In [9]:
# Objective: total shipping cost + depot fixed costs + expansion cost
model += (
    pulp.lpSum(fd_cost[a] * flow_fd[a] for a in arcs_fd) +
    pulp.lpSum(dc_cost[a] * flow_dc[a] for a in arcs_dc) +
    pulp.lpSum(direct_cost[a] * flow_direct[a] for a in arcs_direct) +
    pulp.lpSum(opencost[r] * open_d[r] for r in optional_depots) +
    expand_cost * expand
)

**Factory output** — total flow (via depots + direct) leaving a factory cannot exceed its supply.

In [10]:
for m in markets:
    model += (
        pulp.lpSum(flow_fd[a] for a in arcs_fd if a[0] == m) +
        pulp.lpSum(flow_direct[a] for a in arcs_direct if a[0] == m)
        <= supply[m]
    ), f'factory_supply_{m}'

**Depot flow conservation** — everything that flows into a depot must flow back out to a customer.

In [11]:
for r in regions:
    model += (
        pulp.lpSum(flow_fd[a] for a in arcs_fd if a[1] == r) ==
        pulp.lpSum(flow_dc[a] for a in arcs_dc if a[0] == r)
    ), f'depot_conservation_{r}'

**Depot capacity** — capped at throughput, gated by whether the depot is open. Western Europe's cap can also grow by 20% if `expand = 1`.

In [12]:
for r in regions:
    inflow = pulp.lpSum(flow_fd[a] for a in arcs_fd if a[1] == r)
    if r == expand_depot:
        model += inflow <= through[r] * (1 + expand_pct * expand), f'depot_capacity_{r}'
    else:
        model += inflow <= through[r] * open_d[r], f'depot_capacity_{r}'

**Customer demand** — each customer's demand must be met, either via its depot or the direct route.

In [13]:
for c in customers:
    model += (
        pulp.lpSum(flow_dc[a] for a in arcs_dc if a[1] == c) +
        pulp.lpSum(flow_direct[a] for a in arcs_direct if a[1] == c)
        == demand[c]
    ), f'customer_demand_{c}'

**Depot count** — at most 4 of the 6 depots may remain open.

In [14]:
model += pulp.lpSum(open_d[r] for r in regions) <= 4, 'depot_count'

We now solve the model.

In [15]:
status = model.solve(pulp.PULP_CBC_CMD(msg=0))
print('Solve status  :', pulp.LpStatus[status])
print('Total cost ($):', round(pulp.value(model.objective), 2))

Solve status  : Optimal
Total cost ($): 25898002.0


---
## Analysis

The results below answer the three business questions posed at the start of the notebook.

In [16]:
open_list = [r for r in regions if open_d[r].value() > 0.5]
closed_list = [r for r in regions if r not in open_list]
print('Depots kept open  :', open_list)
print('Depots closed      :', closed_list)
print('Expand Western Europe?', 'Yes' if expand.value() > 0.5 else 'No')

Depots kept open  : ['Western Europe', 'Northern Europe', 'Central America', 'South America']
Depots closed      : ['Southern Europe', 'Caribbean']
Expand Western Europe? No


In [17]:
rows = []
for a in arcs_fd:
    if flow_fd[a].value() and flow_fd[a].value() > 1e-6:
        rows.append({'From': a[0], 'To': a[1], 'Flow (units)': round(flow_fd[a].value())})
for a in arcs_dc:
    if flow_dc[a].value() and flow_dc[a].value() > 1e-6:
        rows.append({'From': a[0], 'To': a[1], 'Flow (units)': round(flow_dc[a].value())})
for a in arcs_direct:
    if flow_direct[a].value() and flow_direct[a].value() > 1e-6:
        rows.append({'From': a[0] + ' (direct)', 'To': a[1], 'Flow (units)': round(flow_direct[a].value())})

product_flow = pd.DataFrame(rows)
product_flow.index = [''] * len(product_flow)
product_flow

,From,To,Flow (units)
,Europe,Northern Europe,15200
,Europe,Western Europe,27700
,LATAM,Central America,28800
,LATAM,South America,17400
,Central America,México,28800
,Northern Europe,Reino Unido,15200
,South America,Brasil,17400
,Western Europe,Francia,27700
,Europe (direct),Italia,10200
,LATAM (direct),República Dominicana,8100


### Interpretation

- **Which four depots stay open?** The optimizer keeps the two forced depots (**Western Europe**, **Central America**) plus whichever optional depots deliver the best trade-off between their fixed operating cost and the much higher cost of shipping their customer directly from the factory. In this run that means **Northern Europe** and **South America** remain open, while **Southern Europe** and **Caribbean** are closed — their customers (Italia, República Dominicana) are served by the direct/express route instead, because their demand volumes are small enough that paying the 30% shipping premium is still cheaper than keeping a low-volume depot open.
- **Should Western Europe be expanded?** No — Western Europe's throughput capacity comfortably covers Francia's demand on its own, so the $15,000 expansion cost buys no benefit and the optimizer leaves `expand = 0`.
- **Which routes satisfy demand?** México, Francia, Brasil, and Reino Unido are served through their regional depots (the cost-efficient path); Italia and República Dominicana are served directly from their factory, since their depots were the two closed in this network redesign.

**Caveat:** because DataCo's raw data doesn't include real facility or direct-shipping costs, the depot fixed costs and the direct-shipping premium above are modeling assumptions (documented in Step 3) built from the closest available proxies in the dataset (discount spend, and a percentage markup on the two-leg cost). Swap in your own facility-cost and freight-premium estimates to tailor the model to an actual network redesign decision.

---
## References

- H. Paul Williams, *Model Building in Mathematical Programming*, fifth edition — original Supply Network Design formulation this model is adapted from.
- `DataCoSupplyChainDataset.csv` — DataCo Global Supply Chain dataset (Kaggle), used here to derive all capacity, demand, and cost inputs.